# PromptWar on Google Colab

Run the **PromptWar** OpenEnv environment + trainer on Colab end-to-end.

What this notebook does:
1. Clones the repo and installs deps (env + optional trainer)
2. Runs the pure-Python test suite to verify the install
3. Starts the FastAPI env server in the background (with optional Consumer Model on GPU)
4. Drives rollouts via `PromptWarEnv` and the `training/` CLI (`smoke` / `live` / `baseline` / `long`)

**Recommended runtime:** `Runtime > Change runtime type > GPU` (T4 is enough for the 0.5B Consumer Model; an A100/L4 is needed for the 3B trainer base).  
CPU-only Colab also works for `--mode smoke`, `--mode baseline` (stub env), and rubric-fallback rollouts.

## 1. Clone the repo

In [1]:
%cd /content
![ -d meta-hackathon ] || git clone https://github.com/rishabhshukla0912/meta-hackathon.git
%cd /content/meta-hackathon
!git pull --ff-only || true
!ls

/content
Cloning into 'meta-hackathon'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 135 (delta 33), reused 132 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 536.14 KiB | 10.51 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/meta-hackathon
Already up to date.
'[External] Meta OpenEnv Hackathon Participant Help Guide.pdf'	 README.md.pdf
 PromptWar_env


In [3]:
!git checkout jyotsna_version_1

Branch 'jyotsna_version_1' set up to track remote branch 'jyotsna_version_1' from 'origin'.
Switched to a new branch 'jyotsna_version_1'


In [4]:
!git branch

* jyotsna_version_1
  main


### (Alternative) Upload your local copy

If you'd rather upload the working tree from your laptop instead of pulling from GitHub, zip the project locally:

```bash
cd ~/Desktop/Projects && zip -r meta-hackathon.zip meta-hackathon -x '*/.venv/*' '*/__pycache__/*' '*/.git/*'
```

Then run the cell below and pick the zip when prompted. (Skip this cell if you cloned from GitHub above.)

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # pick meta-hackathon.zip
# !rm -rf /content/meta-hackathon && unzip -q meta-hackathon.zip -d /content && ls /content/meta-hackathon
# %cd /content/meta-hackathon

## 2. Install dependencies

We install in two layers:
- **Env layer** (always): `openenv-core`, FastAPI/uvicorn, httpx, pydantic, regex, transformers/tokenizers — enough to run rollouts and tests.
- **Consumer layer** (optional, GPU): `torch`, `accelerate` to actually load `Qwen2.5-0.5B-Instruct` for real rubric scoring.
- **Trainer layer** (optional, GPU): `peft`, `trl`, `datasets`, `bitsandbytes` for `--mode long` GRPO training.

Colab already ships `torch`, so we pin to whatever's already installed.

In [5]:
# Env layer — pure-Python deps + OpenEnv runtime
%pip install -q "openenv-core @ git+https://github.com/meta-pytorch/OpenEnv.git"
%pip install -q fastapi 'uvicorn[standard]' httpx pydantic regex 'tokenizers>=0.22' 'transformers>=4.56,<6'
%pip install -q nest_asyncio  # so uvicorn plays nice with Colab's loop

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 30.9 MB/s eta 0:00:00


In [6]:
# Consumer layer — only needed if you want real rubric scoring (uses ~1.5 GB VRAM)
import torch
print('CUDA available:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
%pip install -q 'accelerate>=1.0'

CUDA available: True | device: Tesla T4


In [7]:
# Trainer layer — only needed for --mode long GRPO training (requires GPU + ~16 GB VRAM for 3B base)
# Skip this cell if you only want to drive the env with the scripted/random policy.
%pip install -q 'peft>=0.13' 'trl>=0.11' 'datasets>=3.0' 'bitsandbytes>=0.43'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.2 MB/s eta 0:00:00


## 3. Sanity-check the install with the test suite

These are pure-Python (no GPU, no live env) and should all pass.

In [8]:
%cd /content/meta-hackathon
!PYTHONPATH=. python3 -m unittest discover -s PromptWar_env/tests -v

/content/meta-hackathon
test_empty_append_rejects (test_actions.AppendCaseTableTest.test_empty_append_rejects) ... ok
test_normal_append_applies (test_actions.AppendCaseTableTest.test_normal_append_applies) ... ok
test_over_budget_append_rejects (test_actions.AppendCaseTableTest.test_over_budget_append_rejects) ... ok
test_over_ceiling_append_rejects (test_actions.AppendCaseTableTest.test_over_ceiling_append_rejects) ... ok
test_invalid_regex_rejects (test_actions.DeleteCaseTableTest.test_invalid_regex_rejects) ... ok
test_match_above_floor_applies (test_actions.DeleteCaseTableTest.test_match_above_floor_applies) ... ok
test_match_below_floor_rejects (test_actions.DeleteCaseTableTest.test_match_below_floor_rejects) ... ok
test_no_match_passes_without_rejection (test_actions.DeleteCaseTableTest.test_no_match_passes_without_rejection) ... ok
test_parses_append (test_actions.ParseCommandTest.test_parses_append) ... ok
test_parses_del_alias (test_actions.ParseCommandTest.test_parses_del_al

## 4. Start the env server in the background

The server lives at `PromptWar_env/server/app.py`. Set `PROMPTWAR_LOAD_CONSUMER_MODEL=1` if you want the Consumer Model loaded eagerly at startup — otherwise leave it off and either skip the model (rubrics fall back to deterministic heuristics) or load it on demand via `POST /consumer/load`.

We launch it under `nohup` so it survives across cells, and tail the log.

In [14]:
import os, subprocess, time, signal, pathlib

ROOT = pathlib.Path('/content/meta-hackathon')
LOG = ROOT / 'server.log'
PIDFILE = ROOT / 'server.pid'

# Kill any previous instance from this notebook
if PIDFILE.exists():
    try:
        os.kill(int(PIDFILE.read_text().strip()), signal.SIGTERM)
    except ProcessLookupError:
        pass
    PIDFILE.unlink()

env = os.environ.copy()
env['PYTHONPATH'] = str(ROOT)
# Flip to '1' to eagerly load Qwen2.5-0.5B-Instruct at startup (needs GPU + ~1.5 GB VRAM).
env['PROMPTWAR_LOAD_CONSUMER_MODEL'] = '0'

with open(LOG, 'wb') as logf:
    proc = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'PromptWar_env.server.app:app',
         '--host', '127.0.0.1', '--port', '8000'],
        cwd=str(ROOT), env=env, stdout=logf, stderr=subprocess.STDOUT,
    )
PIDFILE.write_text(str(proc.pid))
print(f'started uvicorn pid={proc.pid}, log={LOG}')

# Wait for /state to come up
import httpx
for i in range(30):
    try:
        r = httpx.get('http://127.0.0.1:8000/state', timeout=2.0)
        if r.status_code < 500:
            print('server is up:', r.status_code)
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print('server did not come up — see tail below')

!tail -n 40 {LOG}

started uvicorn pid=3019, log=/content/meta-hackathon/server.log
server is up: 200
INFO:     Started server process [3019]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:40052 - "GET /state HTTP/1.1" 200 OK
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [13]:
# Optional: load the Consumer Model on demand (only meaningful with a GPU runtime)
import httpx
print(httpx.post('http://127.0.0.1:8000/consumer/load', timeout=600.0).json())
print(httpx.get('http://127.0.0.1:8000/consumer/status').json())

{'available': True, 'load_error': ''}
{'available': True, 'load_error': '', 'model_id': 'Qwen/Qwen2.5-0.5B-Instruct'}


## 5. Drive a rollout from the client

Round-trips through the OpenEnv HTTP API: `reset()` → `step(...)` × N. Confirms the env is reachable and turn-taking works.

In [15]:
import sys; sys.path.insert(0, '/content/meta-hackathon')
from PromptWar_env import PromptWarAction, PromptWarEnv

with PromptWarEnv(base_url='http://127.0.0.1:8000') as env:
    env.set_curriculum_stage(1)  # warm-up: 1 round, lenient grading
    result = env.reset()
    print('initial active_agent =', result.observation.active_agent)

    for cmd in [
        'APPEND: Always cite a source.',
        'APPEND: Refuse harmful asks.',
        'APPEND: Aim for ~50 tokens.',
    ]:
        result = env.step(PromptWarAction(command=cmd))
        print(f'after {cmd!r:40s}  next={result.observation.active_agent}  rejected={result.observation.edit_rejected}')

    print('last_rewards =', result.observation.last_rewards)
    print('shared_prompt =', result.observation.shared_prompt)

initial active_agent = A
after 'APPEND: Always cite a source.'           next=S  rejected=False
after 'APPEND: Refuse harmful asks.'            next=B  rejected=False
after 'APPEND: Aim for ~50 tokens.'             next=A  rejected=False
last_rewards = {'A': 4.4, 'S': 3.4, 'B': 1.6999999999999997}
shared_prompt = You are a careful general-purpose assistant. Answer the user's request directly, truthfully, and calmly. Prefer useful concise explanations, avoid hidden agendas, refuse unsafe requests, and preserve factual accuracy. When uncertain, say what is uncertain. Keep responses readable, balanced, and appropriate for a broad audience without optimizing for any single private objective. Always cite a source. Refuse harmful asks. Aim for ~50 tokens.


## 6. Trainer-side smoke test (no GPU, no live env)

Stub env + scripted policy, 5 "GRPO" steps. Verifies trainer wiring before pulling in real models. The actual GRPO step gets skipped if `torch`/`trl` aren't importable — the goal here is just rollout shape.

In [16]:
%cd /content/meta-hackathon
!PYTHONPATH=. python3 -m training.train --mode smoke --steps 5

/content/meta-hackathon
INFO training.train: smoke episode 0: rewards={'A': 8.77408552155776, 'S': 4.242104010080478, 'B': 6.868253127386505}
INFO training.train: smoke episode 1: rewards={'A': 3.006373348625753, 'S': 3.1895812376547426, 'B': 8.722527048793971}
INFO training.train: smoke episode 2: rewards={'A': 12.323914619364173, 'S': 3.4167191065433986, 'B': 5.270222975123687}
INFO training.train: smoke episode 3: rewards={'A': 9.384802070436074, 'S': 9.778802651248157, 'B': 8.459973770474093}
INFO training.train: smoke episode 4: rewards={'A': 8.17327542580652, 'S': 8.062397760906384, 'B': 4.936616304450335}
INFO httpx: HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-3B-Instruct/aa8e72537993ba99e69dfaafa59ed015b17504d1/config.json "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: GET https://huggingface.co/api/resolve-ca

## 7. Live episode against the running env

In [17]:
%cd /content/meta-hackathon
!PYTHONPATH=. python3 -m training.train --mode live --env-url http://127.0.0.1:8000 --episodes 1

/content/meta-hackathon
INFO httpx: HTTP Request: POST http://127.0.0.1:8000/reset "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:8000/step "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:8000/step "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:8000/step "HTTP/1.1 200 OK"
INFO training.train: live episode 0: rewards={'A': 4.4, 'S': 4.0, 'B': 0.7000000000000001}, final_prompt_tokens=None
--- result ---
mode: live
episodes: 1
transitions_per_role: {'A': 1, 'S': 1, 'B': 1}
rewards_per_role: {'A': 4.4, 'S': 4.0, 'B': 0.7000000000000001}
edit_history: []


## 8. Random-policy baseline (sanity-check env difficulty)

30 episodes with a random policy — the env should _not_ be trivially solvable.

In [18]:
%cd /content/meta-hackathon
!PYTHONPATH=. python3 -m training.train --mode baseline --env-url http://127.0.0.1:8000 --episodes 10

/content/meta-hackathon
INFO httpx: HTTP Request: POST http://127.0.0.1:8000/reset "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:8000/step "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:8000/step "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:8000/step "HTTP/1.1 200 OK"
INFO training.train: baseline ep 0: rewards={'A': 3.8, 'S': 3.2, 'B': 3.2}
INFO httpx: HTTP Request: POST http://127.0.0.1:8000/reset "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:8000/step "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:8000/step "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:8000/step "HTTP/1.1 200 OK"
INFO training.train: baseline ep 1: rewards={'A': 3.8, 'S': 3.4, 'B': 1.7}
INFO httpx: HTTP Request: POST http://127.0.0.1:8000/reset "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:8000/step "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:8000/step "HTTP

In [20]:
!curl -X POST http://127.0.0.1:8000/consumer/load

{"available":true,"load_error":""}

In [21]:
!curl http://127.0.0.1:8000/consumer/status

{"available":true,"load_error":"","model_id":"Qwen/Qwen2.5-0.5B-Instruct"}

In [22]:
!PYTHONPATH=. python3 -m training.train --mode baseline --env-url http://127.0.0.1:8000 --episodes 10

INFO httpx: HTTP Request: POST http://127.0.0.1:8000/reset "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:8000/step "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: POST http://127.0.0.1:8000/step "HTTP/1.1 200 OK"
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/usr/local/lib/python3.12/dist-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/httpcore/_sync/connection_pool.py", line 256, in handle_request
    raise exc from None
  File "/usr/local/lib/python3.12/dist-packages/httpcore/_sync/connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/httpcore/_sync/connection.py", line 103

## 9. (GPU) Long GRPO run

**Requires a GPU runtime.** Loads Qwen2.5-3B-Instruct with three named LoRA adapters (`A`, `S`, `B`) and runs `--steps` GRPO iterations against the live env, checkpointing every 100 steps to `./checkpoints/promptwar`.

Start small (e.g. `--steps 50`) before committing to a long run. On a single T4 you'll likely need `--load-in-4bit` to fit the 3B base.

In [19]:
%cd /content/meta-hackathon
# Smaller burn-in run; bump --steps once you're happy with the metrics.
!PYTHONPATH=. python3 -m training.train --mode long \
    --env-url http://127.0.0.1:8000 \
    --steps 50 \
    --load-in-4bit \
    --checkpoint-every 25 \
    --output-dir ./checkpoints/promptwar

/content/meta-hackathon
INFO httpx: HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-3B-Instruct/aa8e72537993ba99e69dfaafa59ed015b17504d1/config.json "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-3B-Instruct/aa8e72537993ba99e69dfaafa59ed015b17504d1/tokenizer_config.json "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-3B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "

## 10. Stop the env server

In [ ]:
import os, signal, pathlib
PIDFILE = pathlib.Path('/content/meta-hackathon/server.pid')
if PIDFILE.exists():
    pid = int(PIDFILE.read_text().strip())
    try:
        os.kill(pid, signal.SIGTERM)
        print(f'stopped uvicorn pid={pid}')
    except ProcessLookupError:
        print(f'pid {pid} already gone')
    PIDFILE.unlink()
else:
    print('no server.pid found')

### Tips

- **Tail the server log live:** `!tail -f /content/meta-hackathon/server.log` (interrupt the cell to stop tailing).
- **Persist checkpoints across Colab sessions:** mount Drive (`from google.colab import drive; drive.mount('/content/drive')`) and pass `--output-dir /content/drive/MyDrive/promptwar-ckpts`.
- **Curriculum stages:** call `env.set_curriculum_stage(1|2|3)` before `reset()`. Stage 1 = warm-up (1 round, lenient), 2 = standard (3 rounds), 3 = strict.
- **Action grammar:** `APPEND: <text>`, `DELETE: <regex>`, `REPLACE: <old> --> <new>`, `PASS`. See `PromptWar_env/README.md` §4.3 for the full edit-case table.